In [ ]:
import requests
import os
from dotenv import load_dotenv
load_dotenv()
url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"
response = requests.get(url, headers = {"Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}"})
response.status_code

In [8]:
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/8340',
  'repository_url': 'https://api.github.com/repos/huggingface/datasets',
  'labels_url': 'https://api.github.com/repos/huggingface/datasets/issues/8340/labels{/name}',
  'comments_url': 'https://api.github.com/repos/huggingface/datasets/issues/8340/comments',
  'events_url': 'https://api.github.com/repos/huggingface/datasets/issues/8340/events',
  'html_url': 'https://github.com/huggingface/datasets/pull/8340',
  'id': 4908511747,
  'node_id': 'PR_kwDODunzps7y1YDI',
  'number': 8340,
  'title': 'Make Dataset generic to allow specifying column types in typehints',
  'user': {'login': 'RudrenduPaul',
   'id': 38769913,
   'node_id': 'MDQ6VXNlcjM4NzY5OTEz',
   'avatar_url': 'https://avatars.githubusercontent.com/u/38769913?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/RudrenduPaul',
   'html_url': 'https://github.com/RudrenduPaul',
   'followers_url': 'https://api.github.com/users/RudrenduPaul/

In [6]:
import time
import math
import requests
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv()

def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=10_000,
    rate_limit=5_000,
    issues_path=Path("."),
):
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    headers = {"Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}"}
    batch = []
    all_issues = []
    per_page = 100  ## 每页返回的 issue 的数量
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    for page in tqdm(range(num_pages)):
        # 使用 state=all 进行查询来获取 open 和 closed 的issue
        query = f"issues?page={page}&per_page={per_page}&state=all"
        issues = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)
        batch.extend(issues.json())

        if len(batch) > rate_limit and len(all_issues) < num_issues:
            all_issues.extend(batch)
            batch = []  # 重置batch
            print(f"Reached GitHub rate limit. Sleeping for one hour ...")
            time.sleep(60 * 60 + 1)

    all_issues.extend(batch)
    df = pd.DataFrame.from_records(all_issues)
    df.to_json(f"{issues_path}/{repo}-issues.jsonl", orient="records", lines=True)
    print(
        f"Downloaded all the issues for {repo}! Dataset stored at {issues_path}/{repo}-issues.jsonl"
    )

In [7]:
fetch_issues(rate_limit=10_000)

  0%|          | 0/100 [00:00<?, ?it/s]

Downloaded all the issues for datasets! Dataset stored at ./datasets-issues.jsonl


In [9]:
import pandas as pd
from datasets import Dataset

df = pd.read_json("datasets-issues.jsonl", lines=True)
issues_dataset = Dataset.from_pandas(df)
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment'],
    num_rows: 8224
})

In [10]:
sample = issues_dataset.shuffle(seed=666).select(range(3))

In [11]:
for url, pr in zip(sample["html_url"], sample["pull_request"]):
    print(f">> URL: {url}")
    print(f">> Pull request: {pr}\n")

>> URL: https://github.com/huggingface/datasets/issues/4775
>> Pull request: None

>> URL: https://github.com/huggingface/datasets/pull/3529
>> Pull request: {'diff_url': 'https://github.com/huggingface/datasets/pull/3529.diff', 'html_url': 'https://github.com/huggingface/datasets/pull/3529', 'merged_at': '2022-01-05T12:50:14Z', 'patch_url': 'https://github.com/huggingface/datasets/pull/3529.patch', 'url': 'https://api.github.com/repos/huggingface/datasets/pulls/3529'}

>> URL: https://github.com/huggingface/datasets/pull/4414
>> Pull request: {'diff_url': 'https://github.com/huggingface/datasets/pull/4414.diff', 'html_url': 'https://github.com/huggingface/datasets/pull/4414', 'merged_at': '2022-05-31T14:58:51Z', 'patch_url': 'https://github.com/huggingface/datasets/pull/4414.patch', 'url': 'https://api.github.com/repos/huggingface/datasets/pulls/4414'}



In [12]:
issues_dataset = issues_dataset.map(
    lambda x: {"is_pull_request": False if x["pull_request"] is None else True}
)

Map:   0%|          | 0/8224 [00:00<?, ? examples/s]

In [14]:
issue_number = 2792
headers = {"Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}"}
url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
response = requests.get(url, headers=headers)
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/897594128',
  'html_url': 'https://github.com/huggingface/datasets/pull/2792#issuecomment-897594128',
  'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/2792',
  'id': 897594128,
  'node_id': 'IC_kwDODunzps41gDMQ',
  'user': {'login': 'bhavitvyamalik',
   'id': 19718818,
   'node_id': 'MDQ6VXNlcjE5NzE4ODE4',
   'avatar_url': 'https://avatars.githubusercontent.com/u/19718818?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/bhavitvyamalik',
   'html_url': 'https://github.com/bhavitvyamalik',
   'followers_url': 'https://api.github.com/users/bhavitvyamalik/followers',
   'following_url': 'https://api.github.com/users/bhavitvyamalik/following{/other_user}',
   'gists_url': 'https://api.github.com/users/bhavitvyamalik/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/bhavitvyamalik/starred{/owner}{/repo}',
   'subscriptions_url': 'https://api.github.com/users/

In [30]:
def get_comments(issue_number):
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
    response = requests.get(url, headers=headers)
    return [r["body"] for r in response.json()]

get_comments(2792)

["@albertvillanova my tests are failing here:\r\n```\r\ndataset_name = 'gooaq'\r\n\r\n    def test_load_dataset(self, dataset_name):\r\n        configs = self.dataset_tester.load_all_configs(dataset_name, is_local=True)[:1]\r\n>       self.dataset_tester.check_load_dataset(dataset_name, configs, is_local=True, use_local_dummy_data=True)\r\n\r\ntests/test_dataset_common.py:234: \r\n_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ \r\ntests/test_dataset_common.py:187: in check_load_dataset\r\n    self.parent.assertTrue(len(dataset[split]) > 0)\r\nE   AssertionError: False is not true\r\n```\r\nWhen I try loading dataset on local machine it works fine. Any suggestions on how can I avoid this error?",
 'Thanks for the help, @albertvillanova! All tests are passing now.']

In [27]:
issues_with_comments_dataset = issues_dataset.select(range(1000)).map(
    lambda x: {"comments": get_comments(x["number"])}
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [28]:
issues_with_comments_dataset.push_to_hub("github-issues")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Kate-lf/github-issues/commit/203e4747d3a3910b2c92d3547cafab165f1cd197', commit_message='Upload dataset', commit_description='', oid='203e4747d3a3910b2c92d3547cafab165f1cd197', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kate-lf/github-issues', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kate-lf/github-issues'), pr_revision=None, pr_num=None)